# Week 8 — Power BI Gold Export (SQL)

## Transaction & Fraud Analytics Project

This notebook is the **SQL version** of `06_powerbi_export.ipynb`.

It uses the approved Week-7 Gold tables created by `05_gold_aggregations.ipynb`.

### Approved Gold tables

1. `gold_daily_transaction_summary`
2. `gold_merchant_summary`
3. `gold_customer_summary`
4. `gold_fraud_summary`
5. `gold_daily_fraud_summary`
6. `gold_merchant_risk_tier_summary`
7. `gold_merchant_country_summary`
8. `gold_merchant_status_summary`

**Week 8 flow:**

`Gold → Export → Power BI → Reconcile → Evidence`

Power BI must use approved Gold outputs only.

## 1. Select the Gold catalog and schema

Run this first.

In [0]:
USE CATALOG data_engineering;
USE SCHEMA default;

## 2. Verify all approved Gold tables exist

This query checks the tables created by your Week-7 Gold notebook.

In [0]:
SHOW TABLES IN data_engineering.default;

database,tableName,isTemporary
default,bronze_customers,false
default,bronze_devices,false
default,bronze_fraud_cases,false
default,bronze_merchants,false
default,bronze_transactions,false
default,dq_customer_accounts_results,false
default,dq_customers_results,false
default,dq_devices_results,false
default,dq_fraud_cases_results,false
default,dq_merchants_results,false


## 3. Gold source register

The dashboard can use one or more of these Gold tables. Do not assume one table equals one visual.

In [0]:
SELECT 'gold_daily_transaction_summary' AS gold_table,
       'Daily transaction reporting' AS business_purpose,
       'KPI and trend' AS power_bi_use
UNION ALL
SELECT 'gold_merchant_summary',
       'Merchant transaction performance and risk',
       'Merchant comparison'
UNION ALL
SELECT 'gold_customer_summary',
       'Customer transaction activity and risk',
       'Customer comparison'
UNION ALL
SELECT 'gold_fraud_summary',
       'Overall fraud-case summary',
       'Fraud KPI cards'
UNION ALL
SELECT 'gold_daily_fraud_summary',
       'Daily fraud-case trend',
       'Fraud trend'
UNION ALL
SELECT 'gold_merchant_risk_tier_summary',
       'Transaction activity by merchant risk tier',
       'Risk-tier comparison'
UNION ALL
SELECT 'gold_merchant_country_summary',
       'Transaction activity by merchant country',
       'Country comparison'
UNION ALL
SELECT 'gold_merchant_status_summary',
       'Transaction activity by merchant status',
       'Status comparison';

gold_table,business_purpose,power_bi_use
gold_daily_transaction_summary,Daily transaction reporting,KPI and trend
gold_merchant_summary,Merchant transaction performance and risk,Merchant comparison
gold_customer_summary,Customer transaction activity and risk,Customer comparison
gold_fraud_summary,Overall fraud-case summary,Fraud KPI cards
gold_daily_fraud_summary,Daily fraud-case trend,Fraud trend
gold_merchant_risk_tier_summary,Transaction activity by merchant risk tier,Risk-tier comparison
gold_merchant_country_summary,Transaction activity by merchant country,Country comparison
gold_merchant_status_summary,Transaction activity by merchant status,Status comparison


## 4. Check row counts for every Gold table

These are the source counts that should be compared with the exported files.

In [0]:
SELECT 'gold_daily_transaction_summary' AS gold_table,
       COUNT(*) AS row_count
FROM gold_daily_transaction_summary
UNION ALL
SELECT 'gold_merchant_summary', COUNT(*)
FROM gold_merchant_summary
UNION ALL
SELECT 'gold_customer_summary', COUNT(*)
FROM gold_customer_summary
UNION ALL
SELECT 'gold_fraud_summary', COUNT(*)
FROM gold_fraud_summary
UNION ALL
SELECT 'gold_daily_fraud_summary', COUNT(*)
FROM gold_daily_fraud_summary
UNION ALL
SELECT 'gold_merchant_risk_tier_summary', COUNT(*)
FROM gold_merchant_risk_tier_summary
UNION ALL
SELECT 'gold_merchant_country_summary', COUNT(*)
FROM gold_merchant_country_summary
UNION ALL
SELECT 'gold_merchant_status_summary', COUNT(*)
FROM gold_merchant_status_summary;

gold_table,row_count
gold_daily_transaction_summary,181
gold_merchant_summary,2800
gold_customer_summary,18000
gold_fraud_summary,1
gold_daily_fraud_summary,183
gold_merchant_risk_tier_summary,3
gold_merchant_country_summary,5
gold_merchant_status_summary,3


## 5. Inspect the Gold tables

Run these queries to verify the fields before exporting them.

In [0]:
SELECT * FROM gold_daily_transaction_summary;

transaction_date,total_transactions,total_amount_reporting_inr,avg_amount_reporting_inr,avg_risk_score,high_risk_transactions,gold_created_at
2026-03-22,1479,14552906.25,9839.693205,21.816091954022987,4,2026-09-11T08:30:03.253Z
2026-02-08,1510,14377189.46,9521.317523,22.143046357615894,5,2026-09-11T08:30:03.253Z
2026-05-09,1538,21882106.89,14227.637770,19.87256176853056,2,2026-09-11T08:30:03.253Z
2026-04-13,1578,22401938.25,14196.412072,22.1128010139417,8,2026-09-11T08:30:03.253Z
2026-02-13,1548,7391859.99,4775.103353,22.742248062015506,6,2026-09-11T08:30:03.253Z
2026-05-24,1552,15216109.20,9804.194072,20.52577319587629,1,2026-09-11T08:30:03.253Z
2026-06-30,1585,7312356.48,4613.474120,19.414511041009465,0,2026-09-11T08:30:03.253Z
2026-02-07,1557,7442976.16,4780.331509,22.132305716120744,8,2026-09-11T08:30:03.253Z
2026-04-07,1549,23069208.14,14892.968457,21.40477727566172,6,2026-09-11T08:30:03.253Z
2026-01-13,1491,7360022.31,4936.299336,23.640509725016766,14,2026-09-11T08:30:03.253Z


In [0]:
SELECT * FROM gold_merchant_summary;

merchant_id,merchant_label,merchant_category,merchant_country_code,merchant_risk_tier,onboarding_date,merchant_status,settlement_currency,total_transactions,total_amount_reporting_inr,avg_amount_reporting_inr,avg_risk_score,high_risk_transactions,gold_created_at
MER002363,SYNTH_MERCHANT_02363,EDUCATION,IN,LOW,2022-06-07,ACTIVE,INR,105,600254.65,5716.710952,16.571428571428573,0,2026-09-11T08:30:32.924Z
MER002378,SYNTH_MERCHANT_02378,ENTERTAINMENT,AE,LOW,2022-05-25,ACTIVE,AED,110,292455.75,2658.688636,26.672727272727272,0,2026-09-11T08:30:32.924Z
MER000593,SYNTH_MERCHANT_00593,FUEL,GB,MEDIUM,2024-05-06,ACTIVE,GBP,106,246970.50,2329.910377,34.87735849056604,0,2026-09-11T08:30:32.924Z
MER001686,SYNTH_MERCHANT_01686,TRAVEL,IN,LOW,2021-03-03,ACTIVE,INR,99,1246107.73,12586.946768,13.818181818181818,0,2026-09-11T08:30:32.924Z
MER002235,SYNTH_MERCHANT_02235,ELECTRONICS,IN,LOW,2022-02-14,ACTIVE,INR,95,781330.21,8224.528526,15.51578947368421,0,2026-09-11T08:30:32.924Z
MER000286,SYNTH_MERCHANT_00286,FUEL,IN,LOW,2021-06-15,ACTIVE,INR,88,237706.23,2701.207159,13.909090909090908,0,2026-09-11T08:30:32.924Z
MER001706,SYNTH_MERCHANT_01706,DINING,IN,LOW,2025-09-12,ACTIVE,INR,99,343999.74,3474.744848,16.03030303030303,0,2026-09-11T08:30:32.924Z
MER001949,SYNTH_MERCHANT_01949,ENTERTAINMENT,IN,MEDIUM,2022-01-16,ACTIVE,INR,103,251677.22,2443.468155,25.320388349514563,1,2026-09-11T08:30:32.924Z
MER000329,SYNTH_MERCHANT_00329,FUEL,IN,LOW,2021-08-14,ACTIVE,INR,106,402510.22,3797.266226,16.471698113207548,0,2026-09-11T08:30:32.924Z
MER001943,SYNTH_MERCHANT_01943,UTILITIES,IN,LOW,2023-03-28,ACTIVE,INR,104,318231.58,3059.919038,13.163461538461538,0,2026-09-11T08:30:32.924Z


In [0]:
SELECT * FROM gold_customer_summary;

customer_id,total_transactions,total_amount_reporting_inr,avg_amount_reporting_inr,avg_risk_score,high_risk_transactions,gold_created_at
CUS0013471,14,35522.11,2537.293571,28.0,0,2026-09-11T08:31:02.321Z
CUS0017583,14,44306.65,3164.760714,15.571428571428571,0,2026-09-11T08:31:02.321Z
CUS0009749,28,110540.04,3947.858571,28.392857142857142,0,2026-09-11T08:31:02.321Z
CUS0008251,27,74907.14,2774.338519,17.59259259259259,0,2026-09-11T08:31:02.321Z
CUS0006985,11,38517.11,3501.555455,16.545454545454547,0,2026-09-11T08:31:02.321Z
CUS0012863,13,48426.90,3725.146154,21.46153846153846,0,2026-09-11T08:31:02.321Z
CUS0000105,18,104723.95,5817.997222,19.72222222222222,0,2026-09-11T08:31:02.321Z
CUS0014853,15,78539.65,5235.976667,20.266666666666666,0,2026-09-11T08:31:02.321Z
CUS0015723,25,98793.78,3951.751200,22.36,0,2026-09-11T08:31:02.321Z
CUS0005413,13,80127.23,6163.633077,51.0,1,2026-09-11T08:31:02.321Z


In [0]:
SELECT * FROM gold_fraud_summary;

total_fraud_cases,transaction_linked_cases,avg_case_resolution_hours,gold_created_at
4117,4117,62.7276484010,2026-09-11T08:31:37.243Z


In [0]:
SELECT * FROM gold_daily_fraud_summary;

case_date,total_fraud_cases,linked_transaction_cases,avg_resolution_hours,gold_created_at
2026-05-13,14,14,61.0968161538,2026-09-11T08:32:03.222Z
2026-05-16,15,15,60.5764899091,2026-09-11T08:32:03.222Z
2026-02-13,26,26,66.3413157368,2026-09-11T08:32:03.222Z
2026-02-09,29,29,63.3460301250,2026-09-11T08:32:03.222Z
2026-01-30,29,29,58.3159861000,2026-09-11T08:32:03.222Z
2026-01-24,33,33,65.7241770000,2026-09-11T08:32:03.222Z
2026-06-08,14,14,71.9724998333,2026-09-11T08:32:03.222Z
2026-02-28,29,29,59.1623430435,2026-09-11T08:32:03.222Z
2026-05-05,31,31,52.7332361500,2026-09-11T08:32:03.222Z
2026-06-11,11,11,58.1893332000,2026-09-11T08:32:03.222Z


In [0]:
SELECT * FROM gold_merchant_risk_tier_summary;

merchant_risk_tier,total_transactions,total_amount_reporting_inr,avg_amount_reporting_inr,avg_risk_score,high_risk_transactions,gold_created_at
HIGH,13974,157801011.72,11292.472572,41.37183340489481,423,2026-09-11T08:34:29.283Z
MEDIUM,65725,598249216.17,9102.308348,27.1559528337771,180,2026-09-11T08:34:29.283Z
LOW,198301,1783283772.22,8992.812806,17.789320275742433,50,2026-09-11T08:34:29.283Z


In [0]:
SELECT * FROM gold_merchant_country_summary LIMIT 10;

merchant_country_code,total_transactions,total_amount_reporting_inr,avg_amount_reporting_inr,avg_risk_score,high_risk_transactions,gold_created_at
AE,14035,112363239.04,8005.930819,28.88307801923762,63,2026-09-11T08:35:14.328Z
US,28067,206754871.94,7366.475645,28.688566644101613,121,2026-09-11T08:35:14.328Z
GB,19472,169339797.99,8696.579601,29.66331142152835,132,2026-09-11T08:35:14.328Z
IN,197836,1899328458.08,9600.519916,18.07535029013931,267,2026-09-11T08:35:14.328Z
SG,18590,151547633.06,8152.105060,28.319526627218934,70,2026-09-11T08:35:14.328Z


In [0]:
SELECT * FROM gold_merchant_status_summary LIMIT 10;

merchant_status,total_transactions,total_amount_reporting_inr,avg_amount_reporting_inr,avg_risk_score,high_risk_transactions,gold_created_at
REVIEW,6455,53491135.49,8286.775444,22.451587916343918,20,2026-09-11T08:35:35.784Z
SUSPENDED,1268,14540425.05,11467.212185,22.729495268138802,6,2026-09-11T08:35:35.784Z
ACTIVE,270277,2471302439.57,9143.591351,21.151814619815966,627,2026-09-11T08:35:35.784Z


## 6. Recommended Power BI Gold exports

Each Gold table should be exported separately.

**Important:** This notebook uses SQL `COPY INTO` for controlled CSV hand-off. Your Databricks workspace disabled the public DBFS root, so **do not use `/FileStore`**.

Replace `<YOUR_VOLUME>` below with an actual Unity Catalog Volume available in your workspace.

Example:

`/Volumes/data_engineering/default/<YOUR_VOLUME>/week08_powerbi_exports/`

In [0]:
SHOW VOLUMES IN data_engineering.default;

database,volume_name
default,myvolume_01


### 6A. Daily transaction summary export

In [0]:
INSERT OVERWRITE DIRECTORY
'/Volumes/data_engineering/default/myvolume_01/week08_powerbi_exports/gold_daily_transaction_summary'
USING CSV
OPTIONS (
  'header' = 'true'
)
SELECT *
FROM data_engineering.default.gold_daily_transaction_summary;

### 6B. Merchant summary export

In [0]:
INSERT OVERWRITE DIRECTORY
'/Volumes/data_engineering/default/myvolume_01/week08_powerbi_exports/gold_merchant_summary'
USING CSV
OPTIONS (
  'header' = 'true'
)
SELECT *
FROM data_engineering.default.gold_merchant_summary;

### 6C. Customer summary export

In [0]:
INSERT OVERWRITE DIRECTORY
'/Volumes/data_engineering/default/myvolume_01/week08_powerbi_exports/gold_customer_summary'
USING CSV
OPTIONS (
  'header' = 'true'
)
SELECT *
FROM data_engineering.default.gold_customer_summary;

### 6D. Fraud summary export

In [0]:
INSERT OVERWRITE DIRECTORY
'/Volumes/data_engineering/default/myvolume_01/week08_powerbi_exports/gold_fraud_summary'
USING CSV
OPTIONS (
  'header' = 'true'
)
SELECT *
FROM data_engineering.default.gold_fraud_summary;

### 6E. Daily fraud summary export

In [0]:
INSERT OVERWRITE DIRECTORY
'/Volumes/data_engineering/default/myvolume_01/week08_powerbi_exports/gold_daily_fraud_summary'
USING CSV
OPTIONS (
  'header' = 'true'
)
SELECT *
FROM data_engineering.default.gold_daily_fraud_summary;

### 6F. Merchant risk tier summary export

In [0]:
INSERT OVERWRITE DIRECTORY
'/Volumes/data_engineering/default/myvolume_01/week08_powerbi_exports/gold_merchant_risk_tier_summary'
USING CSV
OPTIONS (
  'header' = 'true'
)
SELECT *
FROM data_engineering.default.gold_merchant_risk_tier_summary;

### 6G. Merchant country summary export

In [0]:
INSERT OVERWRITE DIRECTORY
'/Volumes/data_engineering/default/myvolume_01/week08_powerbi_exports/gold_merchant_country_summary'
USING CSV
OPTIONS (
  'header' = 'true'
)
SELECT *
FROM data_engineering.default.gold_merchant_country_summary;

### 6H. Merchant status summary export

In [0]:
INSERT OVERWRITE DIRECTORY
'/Volumes/data_engineering/default/myvolume_01/week08_powerbi_exports/gold_merchant_status_summary'
USING CSV
OPTIONS (
  'header' = 'true'
)
SELECT *
FROM data_engineering.default.gold_merchant_status_summary;

## 7. Verify the exported files

After running the `COPY INTO` commands, use `LIST` to verify that files were created.

Replace `<YOUR_VOLUME>` with your real Unity Catalog Volume name.

In [0]:
LIST '/Volumes/data_engineering/default/myvolume_01/week08_powerbi_exports/';

path,name,size,modification_time
/Volumes/data_engineering/default/myvolume_01/week08_powerbi_exports/gold_customer_summary/,gold_customer_summary/,0,1790421846873
/Volumes/data_engineering/default/myvolume_01/week08_powerbi_exports/gold_daily_fraud_summary/,gold_daily_fraud_summary/,0,1790421846873
/Volumes/data_engineering/default/myvolume_01/week08_powerbi_exports/gold_daily_transaction_summary/,gold_daily_transaction_summary/,0,1790421846874
/Volumes/data_engineering/default/myvolume_01/week08_powerbi_exports/gold_fraud_summary/,gold_fraud_summary/,0,1790421846874
/Volumes/data_engineering/default/myvolume_01/week08_powerbi_exports/gold_merchant_country_summary/,gold_merchant_country_summary/,0,1790421846874
/Volumes/data_engineering/default/myvolume_01/week08_powerbi_exports/gold_merchant_risk_tier_summary/,gold_merchant_risk_tier_summary/,0,1790421846874
/Volumes/data_engineering/default/myvolume_01/week08_powerbi_exports/gold_merchant_status_summary/,gold_merchant_status_summary/,0,1790421846874
/Volumes/data_engineering/default/myvolume_01/week08_powerbi_exports/gold_merchant_summary/,gold_merchant_summary/,0,1790421846874


## 8. Reconciliation queries

These are the Gold-side values that you should compare with Power BI.

### Transaction KPIs

In [0]:
SELECT
    SUM(total_transactions) AS total_transactions,
    SUM(total_amount_reporting_inr) AS total_amount_reporting_inr,
    AVG(avg_amount_reporting_inr) AS avg_amount_reporting_inr,
    AVG(avg_risk_score) AS avg_risk_score,
    SUM(high_risk_transactions) AS high_risk_transactions
FROM data_engineering.default.gold_daily_transaction_summary;

total_transactions,total_amount_reporting_inr,avg_amount_reporting_inr,avg_risk_score,high_risk_transactions
278000,2539334000.11,9129.5036379171,21.19165304663853,653


In [0]:
SELECT
    SUM(total_transactions) AS total_transactions,
    SUM(total_amount_reporting_inr) AS total_amount,
    SUM(high_risk_transactions) AS high_risk_transactions
FROM data_engineering.default.gold_daily_transaction_summary;

total_transactions,total_amount,high_risk_transactions
278000,2539334000.11,653


### Fraud KPIs

In [0]:
SELECT
    total_fraud_cases,
    transaction_linked_cases,
    avg_case_resolution_hours
FROM data_engineering.default.gold_fraud_summary;

total_fraud_cases,transaction_linked_cases,avg_case_resolution_hours
4117,4117,62.7276484010


In [0]:
SELECT
    SUM(total_fraud_cases) AS total_fraud_cases
FROM data_engineering.default.gold_daily_fraud_summary;

total_fraud_cases
4117


In [0]:
SELECT
    case_date,
    total_fraud_cases
FROM data_engineering.default.gold_daily_fraud_summary
ORDER BY case_date
LIMIT 20;

case_date,total_fraud_cases
2026-01-01,8
2026-01-02,25
2026-01-03,29
2026-01-04,30
2026-01-05,32
2026-01-06,32
2026-01-07,38
2026-01-08,31
2026-01-09,33
2026-01-10,27


## 9. Optional trend queries

Use these to understand what the Power BI trend should show.

In [0]:
SELECT *
FROM data_engineering.default.gold_daily_transaction_summary
ORDER BY transaction_date;

transaction_date,total_transactions,total_amount_reporting_inr,avg_amount_reporting_inr,avg_risk_score,high_risk_transactions,gold_created_at
2026-01-01,1503,7027592.46,4675.710220,23.494344644045242,5,2026-09-11T08:30:03.253Z
2026-01-02,1520,7096268.74,4668.597855,22.71842105263158,7,2026-09-11T08:30:03.253Z
2026-01-03,1524,6994046.13,4589.269114,23.458661417322833,3,2026-09-11T08:30:03.253Z
2026-01-04,1492,14872104.88,9967.898713,23.029490616621985,5,2026-09-11T08:30:03.253Z
2026-01-05,1501,6905702.19,4600.734304,23.43504330446369,11,2026-09-11T08:30:03.253Z
2026-01-06,1530,7509840.87,4908.392725,22.730718954248367,4,2026-09-11T08:30:03.253Z
2026-01-07,1498,14836681.07,9904.326482,22.48931909212283,5,2026-09-11T08:30:03.253Z
2026-01-08,1526,14902430.74,9765.682005,22.728047182175622,5,2026-09-11T08:30:03.253Z
2026-01-09,1477,7791810.97,5275.430582,23.327014218009477,4,2026-09-11T08:30:03.253Z
2026-01-10,1586,15153187.65,9554.342781,22.798234552332914,8,2026-09-11T08:30:03.253Z


In [0]:
SELECT *
FROM data_engineering.default.gold_daily_fraud_summary
ORDER BY case_date;

case_date,total_fraud_cases,linked_transaction_cases,avg_resolution_hours,gold_created_at
2026-01-01,8,8,68.9668056667,2026-09-11T08:32:03.222Z
2026-01-02,25,25,50.4303306667,2026-09-11T08:32:03.222Z
2026-01-03,29,29,50.3395652609,2026-09-11T08:32:03.222Z
2026-01-04,30,30,69.8777778696,2026-09-11T08:32:03.222Z
2026-01-05,32,32,70.8683760385,2026-09-11T08:32:03.222Z
2026-01-06,32,32,70.3186325385,2026-09-11T08:32:03.222Z
2026-01-07,38,38,74.2708332813,2026-09-11T08:32:03.222Z
2026-01-08,31,31,58.2781217619,2026-09-11T08:32:03.222Z
2026-01-09,33,33,57.8753910000,2026-09-11T08:32:03.222Z
2026-01-10,27,27,68.1590403636,2026-09-11T08:32:03.222Z


In [0]:
SELECT total_fraud_cases
FROM data_engineering.default.gold_fraud_summary;

total_fraud_cases
4117


In [0]:
SELECT
    merchant_risk_tier,
    SUM(total_transactions) AS total_transactions
FROM data_engineering.default.gold_merchant_risk_tier_summary
GROUP BY merchant_risk_tier
ORDER BY total_transactions DESC;

merchant_risk_tier,total_transactions
LOW,198301
MEDIUM,65725
HIGH,13974


In [0]:
SELECT SUM(total_transactions) AS total_transactions
FROM data_engineering.default.gold_daily_transaction_summary;

total_transactions
278000


## 10. Suggested first Power BI dashboard

### Fraud & Transaction Risk Overview

**KPI cards**
- Total Transactions
- Total Transaction Amount
- High-Risk Transactions
- Total Fraud Cases

**Trend**
- Daily Transaction Trend

**Comparison**
- Transactions by Merchant Risk Tier

**Additional visual**
- Daily Fraud Cases

**Slicer**
- Date, country, merchant status, or another dimension that exists at the selected Gold grain.

Do not create one visual for every Gold table.

## 11. Week 8 validation checklist

Before submitting:

- [ ] Week-7 Gold tables exist.
- [ ] This notebook is saved as `notebooks/06_powerbi_export.ipynb`.
- [ ] All selected Gold tables are documented.
- [ ] Gold row counts were checked.
- [ ] Each selected Gold table has a separate export.
- [ ] Export files exist in the Unity Catalog Volume.
- [ ] Power BI uses Gold exports only.
- [ ] No raw/Bronze/Silver/Trusted Silver source is connected.
- [ ] At least one KPI card exists.
- [ ] At least one trend/comparison visual exists.
- [ ] A useful slicer exists.
- [ ] At least one important KPI is reconciled against Gold.
- [ ] `dashboard/powerbi_dashboard.pbix` is saved.
- [ ] `screenshots/week08_gold_connection.png` is captured.
- [ ] `screenshots/week08_powerbi_draft.png` is captured.
- [ ] `weekly_logs/week08_log.md` is completed.
- [ ] AI Transparency Note is included.
- [ ] GitHub commit/push is completed.

**Week 8 goal:** first working, Gold-traceable Power BI dashboard — not the final Week-9 insight/refinement work.